In [1]:
# Cell 1: Imports
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

# Constants
RAW_JSON      = "../data/sharepoint_data.json"
PAGES_PICKLE  = "../data/pages_df.pkl"
TFIDF_MATRIX  = "../data/tfidf_matrix.pkl"
TFIDF_VECT    = "../data/tfidf_vectorizer.pkl"

In [2]:
# Cell 2: Load JSON & build DataFrame with correct page_name extraction
with open(RAW_JSON, "r", encoding="utf-8") as f:
    raw = json.load(f)

rows = []
for url, page in raw.items():
    text = page.get("text", "").strip()
    # splitlines() handles any line-ending convention
    lines = text.splitlines()
    # second line (index=1) is the human-readable page title if present
    if len(lines) > 1 and lines[1].strip():
        title = lines[1].strip()
    else:
        # fallback: first non-empty line
        title = next((ln for ln in lines if ln.strip()), "")
    rows.append({
        "page_name":   title,
        "url":         url,
        "description": page.get("description", ""),
        "text":        text
    })

pages_df = pd.DataFrame(rows)
# drop any pages whose text is empty after stripping
pages_df = pages_df[pages_df.text.str.strip().astype(bool)].reset_index(drop=True)

# Peek
print("First 10 page titles:")
print(pages_df.page_name.unique()[:10])
pages_df.head()

First 10 page titles:
['Travel authorisation and booking' 'Safezone' 'Student travel insurance'
 'Disability Service' 'Urgent Help'
 'Counselling and Mental Health Service' 'Meet the team'
 'Consideration of Personal Circumstances' 'Attendance Information'
 'Transfers, Suspensions and Withdrawals']


,page_name,url,description,text
0,Travel authorisation and booking,https://unibradfordac.sharepoint.com/sites/stu...,,Student Travel and Insurance\nTravel authorisa...
1,Safezone,https://unibradfordac.sharepoint.com/sites/hea...,,Health and Safety\nSafezone\n'SafeZone'\nSafeZ...
2,Student travel insurance,https://unibradfordac.sharepoint.com/sites/stu...,,Student Travel and Insurance\nStudent travel i...
3,Disability Service,https://unibradfordac.sharepoint.com/sites/dis...,,Disability Services\nDisability Service\nWelco...
4,Urgent Help,https://unibradfordac.sharepoint.com/sites/cou...,,Counselling and Mental Health Service\nUrgent ...


In [3]:
# Cell 3: Save the DataFrame for quick reloads
pages_df.to_pickle(PAGES_PICKLE)
print(f"✅ Saved {len(pages_df)} pages (with titles) to {PAGES_PICKLE}")

✅ Saved 176 pages (with titles) to ../data/pages_df.pkl


In [4]:
# Cell 4: Compute TF–IDF
vec = TfidfVectorizer(
    max_df=0.8,        # drop very-common tokens
    min_df=2,          # drop extremely rare tokens
    ngram_range=(1,2)  # unigrams + bigrams
)
X = vec.fit_transform(pages_df["text"])
print("TF–IDF matrix shape:", X.shape)

TF–IDF matrix shape: (176, 13176)


In [5]:
# Cell 5: Persist the matrix & vectorizer
with open(TFIDF_MATRIX, "wb") as f:
    pickle.dump(X, f)
with open(TFIDF_VECT, "wb") as f:
    pickle.dump(vec, f)
print("✅ TF–IDF matrix and vectorizer saved.")


✅ TF–IDF matrix and vectorizer saved.
